Train YOLO Models (Experiment 2)

This notebook trains two YOLO models on a free Colab GPU:
1. **14-class YOLO** (Pipeline B) — detects and classifies 14 fruit/veg classes
2. **Objectness YOLO** (Pipeline C) — detects objects without classifying them (1 class)

**Results are saved to:** `Google Drive → SnapShelf/results/` (training logs, charts, weights)

## Step 1: Check GPU & Install Dependencies

In [1]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime to GPU!'}")

Mon Mar  2 17:36:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install ultralytics
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.7 MB/s eta 0:00:00


## Step 2: Mount Google Drive & Extract Datasets

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import os

DRIVE = '/content/drive/MyDrive/SnapShelf'

# Extract 14-class dataset
print("Extracting 14-class dataset...")
with zipfile.ZipFile(f'{DRIVE}/dataset_14class.zip', 'r') as z:
    z.extractall('/content/')
print(f"  Train images: {len(os.listdir('/content/dataset/train/images'))}")
print(f"  Val images:   {len(os.listdir('/content/dataset/val/images'))}")

# Extract objectness dataset
print("\nExtracting objectness dataset...")
with zipfile.ZipFile(f'{DRIVE}/dataset_objectness.zip', 'r') as z:
    z.extractall('/content/')
print(f"  Train images: {len(os.listdir('/content/dataset_objectness/train/images'))}")
print(f"  Val images:   {len(os.listdir('/content/dataset_objectness/val/images'))}")

Extracting 14-class dataset...
  Train images: 19356
  Val images:   2602

Extracting objectness dataset...
  Train images: 19356
  Val images:   2602


## Step 3: Create YOLO Data Configs

In [5]:
# 14-class config
config_14 = """path: /content/dataset
train: train/images
val: val/images

nc: 14
names:
  0: apple
  1: banana
  2: bell_pepper_green
  3: bell_pepper_red
  4: carrot
  5: cucumber
  6: grape
  7: lemon
  8: onion
  9: orange
  10: peach
  11: potato
  12: strawberry
  13: tomato
"""

with open('/content/yolo_14class.yaml', 'w') as f:
    f.write(config_14)
print("Written: /content/yolo_14class.yaml")

# Objectness config
config_obj = """path: /content/dataset_objectness
train: train/images
val: val/images

nc: 1
names:
  0: object
"""

with open('/content/yolo_objectness.yaml', 'w') as f:
    f.write(config_obj)
print("Written: /content/yolo_objectness.yaml")

Written: /content/yolo_14class.yaml
Written: /content/yolo_objectness.yaml


## Step 4: Train 14-Class YOLO (Pipeline B)


In [6]:
from ultralytics import YOLO

# Set seed for reproducibility
import random, numpy as np
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DRIVE = '/content/drive/MyDrive/SnapShelf'

model_14 = YOLO('yolov8s.pt')

# Train directly to Google Drive — weights are saved in real-time,
# so if Colab crashes you still have best.pt and last.pt
results_14 = model_14.train(
    data='/content/yolo_14class.yaml',
    epochs=100,
    batch=-1,           # auto batch size — fills GPU memory optimally
    imgsz=640,
    patience=15,
    lr0=0.01,
    project=f'{DRIVE}/results/yolo_14class',   # save directly to Drive
    name='train',
    exist_ok=True,
    seed=42,
    verbose=True,
    save_period=10,     # extra checkpoint every 10 epochs
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_14class.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8

## Step 5: Train Objectness YOLO (Pipeline C)


In [7]:
# Reset seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

model_obj = YOLO('yolov8s.pt')

# Train directly to Google Drive
results_obj = model_obj.train(
    data='/content/yolo_objectness.yaml',
    epochs=100,
    batch=-1,           # auto batch size
    imgsz=640,
    patience=15,
    lr0=0.01,
    project=f'{DRIVE}/results/yolo_objectness',  # save directly to Drive
    name='train',
    exist_ok=True,
    seed=42,
    verbose=True,
    save_period=10,
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_objectness.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective

## Step 6: Save Weights to Google Drive


In [ ]:
import shutil

DRIVE = '/content/drive/MyDrive/SnapShelf'

# Weights are already on Drive (trained there directly).
# Just copy best.pt to a convenient top-level weights/ folder.
drive_weights = f'{DRIVE}/weights'
os.makedirs(drive_weights, exist_ok=True)

src_14 = f'{DRIVE}/results/yolo_14class/train/weights/best.pt'
dst_14 = f'{drive_weights}/yolo_14class_best.pt'
shutil.copy2(src_14, dst_14)
print(f'Saved: {dst_14}  ({os.path.getsize(dst_14) / 1e6:.1f} MB)')

src_obj = f'{DRIVE}/results/yolo_objectness/train/weights/best.pt'
dst_obj = f'{drive_weights}/yolo_objectness_best.pt'
shutil.copy2(src_obj, dst_obj)
print(f'Saved: {dst_obj}  ({os.path.getsize(dst_obj) / 1e6:.1f} MB)')

print(f'\n=== Your Google Drive SnapShelf folder ===')
print(f'  SnapShelf/')
print(f'  ├── weights/')
print(f'  │   ├── yolo_14class_best.pt  ← download this')
print(f'  │   └── yolo_objectness_best.pt ← download this')
print(f'  └── results/                    (training logs, charts, ALL weights)')
print(f'      ├── yolo_14class/train/')
print(f'      └── yolo_objectness/train/')

## Step 7: Quick Validation

Run a quick test to make sure the models work.

In [9]:
# Test 14-class model on a random val image
import glob
val_images = glob.glob('/content/dataset/val/images/*')
test_img = val_images[0]
print(f'Testing on: {test_img}')

model_test = YOLO(dst_14)
results = model_test.predict(test_img, conf=0.25, verbose=False)
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        name = model_test.names[cls_id]
        print(f'  Detected: {name} ({conf:.2f})')

print('\n14-class model OK!')

# Test objectness model
model_test2 = YOLO(dst_obj)
results2 = model_test2.predict(test_img, conf=0.25, verbose=False)
for r in results2:
    print(f'  Objectness detections: {len(r.boxes)}')

print('Objectness model OK!')
print('\n=== ALL DONE — weights are already safe on Google Drive ===')

Testing on: /content/dataset/val/images/-57_jpg.rf.1c1b061decc85407937852679e17a2d7.jpg
  Detected: bell_pepper_green (0.86)
  Detected: bell_pepper_red (0.81)
  Detected: bell_pepper_green (0.58)
  Detected: bell_pepper_green (0.58)

14-class model OK!
  Objectness detections: 4
Objectness model OK!

=== ALL DONE — weights are already safe on Google Drive ===
